In [4]:
import os
from typing import TypedDict, Annotated, Literal
import operator
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [19]:
load_dotenv()

True

In [20]:
class Review(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review."
    )
class Diagnosis(BaseModel):
    issue_type: str = Field(description="the type of issue.")
    tone: str = Field(description="The tone of the response.")
    urgency: str = Field(description="The urgency of the issue.")

In [24]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [25]:
review = """
I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen.
 I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.
"""

In [26]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.environ.get("gemini_apikey"))
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

In [27]:
llm = model.with_structured_output(Review)
llm2 = model.with_structured_output(Diagnosis)

In [28]:
def find_sentiment(state: ReviewState):
    prompt = f""" Find out the sentiment of the following review. Review: {state['review']}"""
    sent = llm.invoke(prompt).sentiment
    return {"sentiment": sent}

def check_sentiment(state: ReviewState) ->  Literal["pos_response", "run_diagnosis"] :
    if state["sentiment"] == "positive":
        return "pos_response"
    else:
        return "run_diagnosis"

def run_diagnosis(state: ReviewState):
    prompt = f""" Diagnose the following review. And return issue_type, tone and urgency in a structured format.
        Review: {state['review']}"""
    diagnosis = llm2.invoke(prompt)
    return {"diagnosis": diagnosis.model_dump()}

def neg_response(state: ReviewState):
    prompt = f""" In the following review the user had this issue type: {state['diagnosis']['issue_type']}, 
        tone: {state['diagnosis']['tone']}, and urgency: {state['diagnosis']['urgency']}. based on this give halpful suggestions to the user in a warm and empathetic tone.
        Review: {state['review']}"""
    response = model.invoke(prompt).content
    return {"response": response}

def pos_response(state: ReviewState):
    prompt = f""" Write a warm thank you message for the following review. Review: {state['review']}"""
    response = model.invoke(prompt).content
    return {"response": response}

In [29]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("neg_response",neg_response)
graph.add_node("pos_response",pos_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)

graph.add_edge("pos_response", END)

graph.add_edge("run_diagnosis", "neg_response")
graph.add_edge("neg_response", END)

app = graph.compile()

In [31]:
intial_state={
    'review': review,
}
op = app.invoke(intial_state)

In [33]:
op

{'review': '\nI’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen.\n I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.\n',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Login/Authentication failure with app freezing',
  'tone': 'Frustrated and angry',
  'urgency': 'High'},
 'response': 'Hey there,\n\nI’m really sorry you’re stuck on this—having the app freeze right when you need to log in is incredibly frustrating, especially when you’ve already tried reinstalling. Let’s see if we can get you back in as quickly as possible.\n\n---\n\n### 1️⃣ Check the basics first  \n| What to do | Why it helps |\n|------------|--------------|\n| **Restart your device** (power off → wait 30\u202fseconds → power on) | Clears any lingering background processes that might be hanging the app. |\n| **Switch networks** (Wi‑Fi ↔ mobile data, or try a different Wi‑Fi) | A we

In [35]:
op['response']

'Hey there,\n\nI’m really sorry you’re stuck on this—having the app freeze right when you need to log in is incredibly frustrating, especially when you’ve already tried reinstalling. Let’s see if we can get you back in as quickly as possible.\n\n---\n\n### 1️⃣ Check the basics first  \n| What to do | Why it helps |\n|------------|--------------|\n| **Restart your device** (power off → wait 30\u202fseconds → power on) | Clears any lingering background processes that might be hanging the app. |\n| **Switch networks** (Wi‑Fi ↔ mobile data, or try a different Wi‑Fi) | A weak or unstable connection can cause the authentication request to time‑out and freeze the UI. |\n| **Make sure the date & time are set to “Automatic”** | Incorrect system time can break SSL/TLS handshakes, which stops the login flow. |\n\n---\n\n### 2️⃣ Clear app data & cache (even after reinstall)  \n1. **Android**: Settings → Apps → *Your App* → Storage → **Clear Cache** **and** **Clear Data**.  \n2. **iOS**: There’s no